# City Library Management System — Part A

This notebook activates and demonstrates the library system implemented in `library_system.py`.

| Section | What it covers |
|---------|----------------|
| 1 | Import & verify the module |
| 2 | Add books manually |
| 3 | Add members manually |
| 4 | Issue & return books |
| 5 | Borrow log |
| 6 | Reports & queries |
| 7 | Library summary |
| 8 | Full demo with seed data |
| 9 | Run unit tests |

---
**Design decisions:**
- All data is held in three in-memory dicts/lists (`BOOKS`, `MEMBERS`, `BORROW_LOG`).  
- No external database is required — ideal for a standalone demo.  
- All business logic is encapsulated in plain functions (not classes), meeting the *functionalise but do not modularise* requirement.
- Validation is enforced at every entry-point (duplicate IDs, age floor, availability checks).

## 1. Import the library system

In [1]:
import importlib
import library_system as lib
importlib.reload(lib)   # reload ensures a clean state each run

print("library_system loaded successfully.")
print(f"Functions available: {[f for f in dir(lib) if not f.startswith('_')]}")

library_system loaded successfully.
Functions available: ['BOOKS', 'BORROW_LOG', 'Counter', 'MEMBERS', 'add_book', 'add_member', 'datetime', 'display_books', 'display_borrow_log', 'display_members', 'display_summary', 'get_book', 'get_member', 'issue_book', 'library_summary', 'list_members_with_borrowed_books', 'load_seed_data', 'most_popular_genre', 'reset_library', 'return_book', 'search_books', 'show_available_books_by_genre', 'update_book_availability']


## 2. Add books manually

Each book requires a unique **Book ID**, title, author, and genre.  
Availability defaults to `'Available'`.

In [2]:
lib.reset_library()   # start fresh

# Add individual books
b1 = lib.add_book("B001", "The Great Gatsby",        "F. Scott Fitzgerald", "Fiction")
b2 = lib.add_book("B002", "1984",                    "George Orwell",       "Dystopian")
b3 = lib.add_book("B003", "A Brief History of Time", "Stephen Hawking",     "Science")
b4 = lib.add_book("B004", "Dune",                    "Frank Herbert",       "Science Fiction")
b5 = lib.add_book("B005", "The Hobbit",              "J.R.R. Tolkien",      "Fantasy")

print("Books added:")
lib.display_books(list(lib.BOOKS.values()), title="All Books")

Books added:

  All Books  (5 record(s))
------------------------------------------------------------
  [AVAILABLE] B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [AVAILABLE] B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
------------------------------------------------------------


In [3]:
# Demonstrate duplicate-ID guard
try:
    lib.add_book("B001", "Duplicate", "Author", "Genre")
except ValueError as e:
    print(f"Expected error caught: {e}")

Expected error caught: Book ID 'B001' already exists.


## 3. Add members manually

Each member requires a unique **Member ID**, name, age (≥ 5), and contact info.

In [4]:
m1 = lib.add_member("M001", "Alice Johnson", 28, "alice@example.com")
m2 = lib.add_member("M002", "Bob Smith",     35, "bob@example.com")
m3 = lib.add_member("M003", "Carol White",   22, "carol@example.com")

print("Members registered:")
lib.display_members(list(lib.MEMBERS.values()), title="All Members")

Members registered:

  All Members  (3 record(s))
------------------------------------------------------------
  M001 | Alice Johnson | Age: 28 | Contact: alice@example.com | Borrowed: None
  M002 | Bob Smith | Age: 35 | Contact: bob@example.com | Borrowed: None
  M003 | Carol White | Age: 22 | Contact: carol@example.com | Borrowed: None
------------------------------------------------------------


In [5]:
# Demonstrate age guard
try:
    lib.add_member("M099", "Child", 3, "child@example.com")
except ValueError as e:
    print(f"Expected error caught: {e}")

Expected error caught: Member age must be at least 5.


## 4. Issue & Return books

### 4.1 Issue books

In [6]:
# Alice borrows two books
e1 = lib.issue_book("M001", "B001")
e2 = lib.issue_book("M001", "B005")

# Bob borrows one book
e3 = lib.issue_book("M002", "B003")

print("After issuing books:")
lib.display_books(list(lib.BOOKS.values()), title="Book Status")

After issuing books:

  Book Status  (5 record(s))
------------------------------------------------------------
  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [ISSUED]    B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction
  [ISSUED]    B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
------------------------------------------------------------


In [7]:
# Demonstrate: cannot issue an already-issued book
try:
    lib.issue_book("M003", "B001")   # B001 is Issued by Alice
except ValueError as e:
    print(f"Expected error caught: {e}")

Expected error caught: Book 'The Great Gatsby' is already issued and not available.


### 4.2 Return a book

In [8]:
# Bob returns his book
e4 = lib.return_book("M002", "B003")
print(f"Return recorded: {e4}")

print("\nBook status after return:")
lib.display_books(list(lib.BOOKS.values()), title="Book Status")

Return recorded: {'action': 'RETURN', 'member_id': 'M002', 'book_id': 'B003', 'timestamp': '2026-02-17 13:53:44'}

Book status after return:

  Book Status  (5 record(s))
------------------------------------------------------------
  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
  [AVAILABLE] B003 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction
  [ISSUED]    B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
------------------------------------------------------------


In [9]:
# Demonstrate: cannot return a book you do not have
try:
    lib.return_book("M003", "B001")   # Carol never borrowed B001
except ValueError as e:
    print(f"Expected error caught: {e}")

Expected error caught: Member 'Carol White' does not have book 'The Great Gatsby'.


## 5. Borrow log

Every issue and return transaction is recorded with a timestamp.

In [10]:
lib.display_borrow_log()


  BORROW LOG  (4 entry(ies))
------------------------------------------------------------
  [2026-02-17 13:53:44] ISSUE  | Alice Johnson        → The Great Gatsby
  [2026-02-17 13:53:44] ISSUE  | Alice Johnson        → The Hobbit
  [2026-02-17 13:53:44] ISSUE  | Bob Smith            → A Brief History of Time
  [2026-02-17 13:53:44] RETURN | Bob Smith            → A Brief History of Time
------------------------------------------------------------


## 6. Reports & Queries

### 6.1 Available books by genre

In [11]:
for genre in ["Fiction", "Science", "Dystopian", "Fantasy", "Science Fiction"]:
    books = lib.show_available_books_by_genre(genre)
    lib.display_books(books, title=f"Available — {genre}")


  Available — Fiction  (0 record(s))
------------------------------------------------------------
  No records found.
------------------------------------------------------------

  Available — Science  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B003 | A Brief History of Time by Stephen Hawking | Genre: Science
------------------------------------------------------------

  Available — Dystopian  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
------------------------------------------------------------

  Available — Fantasy  (0 record(s))
------------------------------------------------------------
  No records found.
------------------------------------------------------------

  Available — Science Fiction  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B004 | Dune by Frank Herbert | Genre: Science Fiction


### 6.2 Members currently borrowing books

In [12]:
borrowers = lib.list_members_with_borrowed_books()
lib.display_members(borrowers, title="Members with Borrowed Books")


  Members with Borrowed Books  (1 record(s))
------------------------------------------------------------
  M001 | Alice Johnson | Age: N/A | Contact: alice@example.com | Borrowed: The Great Gatsby, The Hobbit
------------------------------------------------------------


### 6.3 Search by title or author

In [13]:
queries = ["gatsby", "orwell", "tolkien", "science"]
for q in queries:
    results = lib.search_books(q)
    lib.display_books(results, title=f'Search: "{q}"')


  Search: "gatsby"  (1 record(s))
------------------------------------------------------------
  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
------------------------------------------------------------

  Search: "orwell"  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B002 | 1984 by George Orwell | Genre: Dystopian
------------------------------------------------------------

  Search: "tolkien"  (1 record(s))
------------------------------------------------------------
  [ISSUED]    B005 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
------------------------------------------------------------

  Search: "science"  (0 record(s))
------------------------------------------------------------
  No records found.
------------------------------------------------------------


### 6.4 Most popular genre

In [14]:
popularity = lib.most_popular_genre()
print(f"Most popular genre : {popularity['genre']} ({popularity['count']} issue(s))")
print()
print("Full genre issue counts:")
for genre, cnt in sorted(popularity["all_genre_counts"].items(), key=lambda x: -x[1]):
    bar = "█" * cnt
    print(f"  {genre:<20} {bar} {cnt}")

Most popular genre : Fiction (1 issue(s))

Full genre issue counts:
  Fiction              █ 1
  Fantasy              █ 1
  Science              █ 1


## 7. Library Summary

In [15]:
summary = lib.library_summary()
lib.display_summary(summary)


  LIBRARY SUMMARY
------------------------------------------------------------
  Books          : 5 total
    Available    : 3
    Issued       : 2
  Genre breakdown:
    Fiction              : 1
    Dystopian            : 1
    Science              : 1
    Science Fiction      : 1
    Fantasy              : 1
  Members        : 3 total
    Borrowing    : 1
  Transactions   : 4
  Popular genre  : Fiction (1 issue(s))
------------------------------------------------------------


## 8. Full demo with seed data

The seed loader populates 15 books, 5 members, and several transactions so every report has rich content to display.

In [16]:
lib.load_seed_data()
print(f"Seed loaded: {len(lib.BOOKS)} books, {len(lib.MEMBERS)} members, "
      f"{len(lib.BORROW_LOG)} log entries.")

Seed loaded: 15 books, 5 members, 9 log entries.


In [17]:
# Full catalogue
lib.display_books(list(lib.BOOKS.values()), title="Full Catalogue")


  Full Catalogue  (15 record(s))
------------------------------------------------------------
  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B002 | To Kill a Mockingbird by Harper Lee | Genre: Fiction
  [ISSUED]    B003 | 1984 by George Orwell | Genre: Dystopian
  [AVAILABLE] B004 | A Brief History of Time by Stephen Hawking | Genre: Science
  [ISSUED]    B005 | Sapiens by Yuval Noah Harari | Genre: History
  [AVAILABLE] B006 | The Selfish Gene by Richard Dawkins | Genre: Science
  [ISSUED]    B007 | Dune by Frank Herbert | Genre: Science Fiction
  [AVAILABLE] B008 | Foundation by Isaac Asimov | Genre: Science Fiction
  [ISSUED]    B009 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
  [AVAILABLE] B010 | Harry Potter & the Philosopher's Stone by J.K. Rowling | Genre: Fantasy
  [AVAILABLE] B011 | The Alchemist by Paulo Coelho | Genre: Fiction
  [AVAILABLE] B012 | Brave New World by Aldous Huxley | Genre: Dystopian
  [ISSUED]    B013 | The Da

In [18]:
# All members
lib.display_members(list(lib.MEMBERS.values()), title="All Registered Members")


  All Registered Members  (5 record(s))
------------------------------------------------------------
  M001 | Alice Johnson | Age: 28 | Contact: alice@example.com | Borrowed: B001, B009
  M002 | Bob Smith | Age: 35 | Contact: bob@example.com | Borrowed: B007
  M003 | Carol White | Age: 22 | Contact: carol@example.com | Borrowed: B003
  M004 | David Brown | Age: 45 | Contact: david@example.com | Borrowed: B005, B013
  M005 | Eva Martinez | Age: 31 | Contact: eva@example.com | Borrowed: B015
------------------------------------------------------------


In [19]:
# Full borrow log
lib.display_borrow_log()


  BORROW LOG  (9 entry(ies))
------------------------------------------------------------
  [2026-02-17 13:53:44] ISSUE  | Alice Johnson        → The Great Gatsby
  [2026-02-17 13:53:44] ISSUE  | Alice Johnson        → The Hobbit
  [2026-02-17 13:53:44] ISSUE  | Bob Smith            → A Brief History of Time
  [2026-02-17 13:53:44] RETURN | Bob Smith            → A Brief History of Time
  [2026-02-17 13:53:44] ISSUE  | Bob Smith            → Dune
  [2026-02-17 13:53:44] ISSUE  | Carol White          → 1984
  [2026-02-17 13:53:44] ISSUE  | David Brown          → Sapiens
  [2026-02-17 13:53:44] ISSUE  | David Brown          → The Da Vinci Code
  [2026-02-17 13:53:44] ISSUE  | Eva Martinez         → Thinking, Fast and Slow
------------------------------------------------------------


In [20]:
# Available books by genre
for genre in ["Fiction", "Science", "Dystopian", "Fantasy", "Science Fiction",
              "History", "Mystery", "Psychology"]:
    books = lib.show_available_books_by_genre(genre)
    if books:
        lib.display_books(books, title=f"Available — {genre}")


  Available — Fiction  (2 record(s))
------------------------------------------------------------
  [AVAILABLE] B002 | To Kill a Mockingbird by Harper Lee | Genre: Fiction
  [AVAILABLE] B011 | The Alchemist by Paulo Coelho | Genre: Fiction
------------------------------------------------------------

  Available — Science  (2 record(s))
------------------------------------------------------------
  [AVAILABLE] B004 | A Brief History of Time by Stephen Hawking | Genre: Science
  [AVAILABLE] B006 | The Selfish Gene by Richard Dawkins | Genre: Science
------------------------------------------------------------

  Available — Dystopian  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B012 | Brave New World by Aldous Huxley | Genre: Dystopian
------------------------------------------------------------

  Available — Fantasy  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B010 | Harry Potter & the Philosoph

In [21]:
# Members currently borrowing
lib.display_members(
    lib.list_members_with_borrowed_books(),
    title="Members with Borrowed Books"
)


  Members with Borrowed Books  (5 record(s))
------------------------------------------------------------
  M001 | Alice Johnson | Age: N/A | Contact: alice@example.com | Borrowed: The Great Gatsby, The Hobbit
  M002 | Bob Smith | Age: N/A | Contact: bob@example.com | Borrowed: Dune
  M003 | Carol White | Age: N/A | Contact: carol@example.com | Borrowed: 1984
  M004 | David Brown | Age: N/A | Contact: david@example.com | Borrowed: Sapiens, The Da Vinci Code
  M005 | Eva Martinez | Age: N/A | Contact: eva@example.com | Borrowed: Thinking, Fast and Slow
------------------------------------------------------------


In [22]:
# Search
for q in ["asimov", "history", "the"]:
    lib.display_books(lib.search_books(q), title=f'Search: "{q}"')


  Search: "asimov"  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B008 | Foundation by Isaac Asimov | Genre: Science Fiction
------------------------------------------------------------

  Search: "history"  (1 record(s))
------------------------------------------------------------
  [AVAILABLE] B004 | A Brief History of Time by Stephen Hawking | Genre: Science
------------------------------------------------------------

  Search: "the"  (6 record(s))
------------------------------------------------------------
  [ISSUED]    B001 | The Great Gatsby by F. Scott Fitzgerald | Genre: Fiction
  [AVAILABLE] B006 | The Selfish Gene by Richard Dawkins | Genre: Science
  [ISSUED]    B009 | The Hobbit by J.R.R. Tolkien | Genre: Fantasy
  [AVAILABLE] B010 | Harry Potter & the Philosopher's Stone by J.K. Rowling | Genre: Fantasy
  [AVAILABLE] B011 | The Alchemist by Paulo Coelho | Genre: Fiction
  [ISSUED]    B013 | The Da Vinci Code by Dan Brown | Genr

In [23]:
# Most popular genre
pop = lib.most_popular_genre()
print(f"Most popular genre : {pop['genre']} ({pop['count']} issue(s))")
print()
print("Genre issue breakdown:")
for genre, cnt in sorted(pop["all_genre_counts"].items(), key=lambda x: -x[1]):
    bar = "█" * cnt
    print(f"  {genre:<20} {bar} {cnt}")

Most popular genre : Fiction (1 issue(s))

Genre issue breakdown:
  Fiction              █ 1
  Fantasy              █ 1
  Science              █ 1
  Science Fiction      █ 1
  Dystopian            █ 1
  History              █ 1
  Mystery              █ 1
  Psychology           █ 1


In [24]:
# Full summary
lib.display_summary(lib.library_summary())


  LIBRARY SUMMARY
------------------------------------------------------------
  Books          : 15 total
    Available    : 8
    Issued       : 7
  Genre breakdown:
    Fiction              : 3
    Dystopian            : 2
    Science              : 2
    History              : 1
    Science Fiction      : 2
    Fantasy              : 2
    Mystery              : 2
    Psychology           : 1
  Members        : 5 total
    Borrowing    : 5
  Transactions   : 9
  Popular genre  : Fiction (1 issue(s))
------------------------------------------------------------


## 9. Run unit tests

The test suite in `test_library_system.py` covers:

| Class | Tests |
|-------|-------|
| `TestAddBook` | add success, duplicate ID, whitespace stripping, get, update availability |
| `TestAddMember` | add success, duplicate ID, age floor, get |
| `TestIssueReturn` | issue success/guards, return success/guards, re-issue after return |
| `TestReports` | available-by-genre, case insensitivity, member list, search, popular genre |
| `TestLibrarySummary` | totals before/after transactions, genre breakdown |
| `TestReset` | all stores cleared |
| `TestSeedData` | book/member counts, log integrity, issued-status consistency |

In [25]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_library_system.py", "-v", "--tb=short"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

============================= test session starts ==============================
platform darwin -- Python 3.12.7, pytest-9.0.2, pluggy-1.6.0 -- /usr/local/bin/python3
cachedir: .pytest_cache
rootdir: /Users/rajeevkulkarni/Library/CloudStorage/OneDrive-Personal/projects/github/pravartak/assignments/week04
plugins: anyio-4.12.1, langsmith-0.6.7
collecting ... collected 47 items

test_library_system.py::TestAddBook::test_add_book_duplicate_id_raises PASSED [  2%]
test_library_system.py::TestAddBook::test_add_book_strips_whitespace PASSED [  4%]
test_library_system.py::TestAddBook::test_add_book_success PASSED        [  6%]
test_library_system.py::TestAddBook::test_get_book_not_found PASSED      [  8%]
test_library_system.py::TestAddBook::test_get_book_success PASSED        [ 10%]
test_library_system.py::TestAddBook::test_update_availability_invalid_status PASSED [ 12%]
test_library_system.py::TestAddBook::test_update_availability_unknown_book PASSED [ 14%]
test_library_system.py::TestAdd

---
## Summary & Assumptions

### What was built
- **Book management** – add, retrieve, and update availability with duplicate-ID protection.
- **Member management** – register members with an age floor of 5; each member tracks their own borrowed list.
- **Issue / Return workflow** – prevents double-borrowing the same book by two members or the same member; updates availability atomically; logs every action with a timestamp.
- **Reports** – available books by genre (case-insensitive), active borrowers, full-text book search, most-popular-genre based on the *complete* issue history (including returned books).
- **Library summary** – single-call snapshot covering totals, genre breakdown, and borrowing stats.

### Design choices
| Choice | Reason |
|--------|--------|
| In-memory dict/list stores | Zero dependencies; easily swapped for a DB later |
| `BORROW_LOG` keeps all entries | Enables historical analytics (popular genre counts returned books too) |
| Functional (not OO/modular) | Matches the assignment constraint |
| `reset_library()` exposed | Enables clean unit-test isolation without reloading the module |
| Age floor = 5 | Prevents data-entry errors; children under 5 would not independently borrow |

### Assumptions
- A member may borrow multiple *different* books simultaneously.
- A book may only be held by one member at a time.
- Popularity is based on total ISSUE events (not current holds), so a popular returned book still counts.
- IDs are strings (e.g., `'B001'`, `'M001'`) and must be unique within their domain.